## Setup

In [6]:
import os
import re

import src.py_src.util as util
import pandas as pd
from dotenv import load_dotenv
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.firefox.service import Service
from selenium.webdriver.firefox.options import Options
from webdriver_manager.firefox import GeckoDriverManager
import time
import ftplib
import tarfile
import shutil
import requests
from bs4 import BeautifulSoup
from urllib.parse import urljoin

In [7]:
load_dotenv()
SAVE_PATH = os.getenv("EVENTS_RAW_PATH")
DATA_RANGE = range(2010, 2024 + 1)

## Events NC

In [ ]:
def get_events_source(date_: pd.Timestamp) -> tuple[str, str]:
    goes13_url = "https://www.ncei.noaa.gov/data/goes-space-environment-monitor/access/science/xrs/goes13/xrsf-l2-flsum_science/"
    goes14_url = "https://www.ncei.noaa.gov/data/goes-space-environment-monitor/access/science/xrs/goes14/xrsf-l2-flsum_science/"
    goes15_url = "https://www.ncei.noaa.gov/data/goes-space-environment-monitor/access/science/xrs/goes15/xrsf-l2-flsum_science/"
    goes16_url = "https://data.ngdc.noaa.gov/platforms/solar-space-observing-satellites/goes/goes16/l2/data/xrsf-l2-flsum_science/"

    # ====================================================================================
    # Alternância baseada na coluna "Science-Quality" da Tabela 3 do artigo "defects_and_inconsistencies" (Seç. 2.3).
    # ====================================================================================
    if date_ >= pd.Timestamp("2017-02-07"):
        return "g16", goes16_url
    elif date_ >= pd.Timestamp("2016-06-09"):
        return "g15", goes15_url
    elif date_ >= pd.Timestamp("2016-05-12"):
        return "g14", goes14_url
    elif date_ >= pd.Timestamp("2015-06-09"):
        return "g13", goes13_url
    elif date_ >= pd.Timestamp("2015-05-21"):
        return "g14", goes14_url
    elif date_ >= pd.Timestamp("2012-11-19"):
        return "g15", goes15_url
    elif date_ >= pd.Timestamp("2012-10-23"):
        return "g14", goes14_url
    elif date_ >= pd.Timestamp("2010-10-28"):
        return "g15", goes15_url
    elif date_ >= pd.Timestamp("2009-12-31"):
        return "g14", goes14_url

    raise ValueError(f"Data fora do intervalo suportado: {date_.date()}")

def download_events_nc_data_selenium(years_range_: range, save_path_: str) -> None:
    save_path_abs = os.path.abspath(save_path_)
    os.makedirs(save_path_abs, exist_ok=True)
    print(f"Arquivos serão salvos em: '{save_path_abs}' (organizados por ano/mês)")

    options = Options()
    options.set_preference("browser.download.folderList", 2)
    options.set_preference("browser.download.dir", save_path_abs)
    options.set_preference("browser.download.manager.showWhenStarting", False)
    options.set_preference("browser.helperApps.neverAsk.saveToDisk", "text/csv,application/octet-stream,application/x-netcdf")

    driver = webdriver.Firefox(service=Service(GeckoDriverManager().install()), options=options)

    try:
        schedule: dict[tuple[int, str], set[str]] = {}
        for y_ in years_range_:
            for date_ in pd.date_range(start=f"{y_}-01-01", end=f"{y_}-12-31"):
                try:
                    _, base_url = get_events_source(date_)
                except ValueError:
                    continue
                schedule.setdefault((y_, base_url), set()).add(f"{date_.month:02d}/")

        for (y_, base_url), target_months in sorted(schedule.items(), key=lambda item: (item[0][0], item[0][1])):

            print(f"Acessando a página principal: {base_url}")
            driver.get(base_url)
            time.sleep(2)

            links = driver.find_elements(By.XPATH, "//table/tbody/tr/td/a")
            year_link = None
            for link in links:
                if link.text.strip() == f"{y_}/":
                    year_link = link
                    break
            if year_link is None:
                print(f"AVISO: Ano {y_} não encontrado em {base_url}")
                continue
            year_link = year_link.get_attribute("href")

            driver.get(year_link)
            time.sleep(1)

            month_elements = driver.find_elements(By.XPATH, "//table/tbody/tr/td/a")
            month_links = {}
            for link in month_elements:
                text = link.text.strip()
                if re.fullmatch(r"\d{2}/", text):
                    month_links[text] = link.get_attribute("href")

            for month_dir in sorted(target_months):
                month_link = month_links.get(month_dir)
                if month_link is None:
                    print(f"  [MÊS] AVISO: Mês {month_dir} não encontrado para {y_} em {base_url}")
                    continue

                month_num = month_dir.rstrip('/')
                month_save_dir = os.path.join(save_path_abs, str(y_), month_num)
                os.makedirs(month_save_dir, exist_ok=True)

                print(f"  [MÊS] Acessando mês: {month_link}")
                driver.get(month_link)
                time.sleep(1)

                day_elements = driver.find_elements(By.XPATH, "//table/tbody/tr/td/a")
                for file in day_elements:
                    if 'Parent Directory' not in file.text:
                        file_name_ = file.text.strip()

                        match_ = re.search(r'_[dm](\d{6,8})_', file_name_)
                        if match_:
                            date_str = match_.group(1)
                            if len(date_str) == 6:
                                date_str += "01"

                            file_date = pd.Timestamp(date_str)
                            expected_sat, _ = get_events_source(file_date)

                            current_sat_match = re.search(r'_g(\d{2})_', file_name_)
                            if current_sat_match:
                                current_sat = current_sat_match.group(1)

                                if expected_sat != f"g{current_sat}":
                                    print(f"        -> Ignorando {file_name_} (Satélite secundário no período)")
                                    continue

                        root_download_file_path = os.path.join(save_path_abs, file_name_)
                        month_file_path = os.path.join(month_save_dir, file_name_)

                        print(f"        -> Baixando {file_name_}...")
                        file.click()
                        util.wait_download(file_path=root_download_file_path, file_name=file_name_)
                        os.replace(root_download_file_path, month_file_path)
    except AttributeError as e:
        print(f"Erro de atributo: {e}")
    except Exception as e:
        print(f"Erro inesperado: {e}")
    finally:
        print("\nProcesso concluído. Fechando o navegador.")
        driver.quit()

    return None

In [ ]:
download_events_nc_data_selenium(DATA_RANGE, SAVE_PATH)

## SWPC FTP

In [ ]:
def download_swpc_ftp_events(years_range_: range, save_path_: str) -> None:
    """
    Conecta ao servidor FTP da SWPC e baixa os arquivos YYYY_events.tar.gz,
    extraindo cada ano para raw/<YYYY>/<YYYY>_SWPC_events.
    """
    save_path_abs = os.path.abspath(save_path_)
    os.makedirs(save_path_abs, exist_ok=True)

    print(f"Arquivos SWPC serão salvos em subpastas anuais de: '{save_path_abs}'")

    ftp_host = "ftp.swpc.noaa.gov"
    base_dir = "/pub/warehouse/"

    try:
        print(f"Estabelecendo conexão FTP com: {ftp_host}...")
        ftp = ftplib.FTP(ftp_host)
        ftp.login() # A NOAA permite login anônimo

        for y_ in years_range_:
            year_dir_ = f"{base_dir}{y_}/"
            expected_filename = f"{y_}_events.tar.gz"

            year_save_dir = os.path.join(save_path_abs, str(y_))
            extracted_dir = os.path.join(year_save_dir, f"{y_}_SWPC_events")
            archive_path = os.path.join(year_save_dir, expected_filename)

            os.makedirs(year_save_dir, exist_ok=True)

            if os.path.isdir(extracted_dir) and os.listdir(extracted_dir):
                print(f"  -> [SKIP] Pasta já extraída: {extracted_dir}")
                continue

            try:
                ftp.cwd(year_dir_)
            except ftplib.error_perm as e:
                print(f"  -> AVISO: Diretório do ano {y_} não encontrado. Erro: {e}")
                continue

            files_in_dir = ftp.nlst()
            if expected_filename not in files_in_dir:
                print(f"  -> AVISO: O arquivo {expected_filename} não foi encontrado na pasta {year_dir_}.")
                continue

            if not os.path.exists(archive_path):
                print(f"  -> Baixando {expected_filename} via FTP...")
                with open(archive_path, "wb") as f:
                    ftp.retrbinary(f"RETR {expected_filename}", f.write)
            else:
                print(f"  -> Reutilizando arquivo já baixado: {archive_path}")

            os.makedirs(extracted_dir, exist_ok=True)
            print(f"  -> Extraindo para: {extracted_dir}")
            with tarfile.open(archive_path, "r:gz") as tar:
                tar.extractall(path=extracted_dir)

            os.remove(archive_path)
            time.sleep(1)

        ftp.quit()
        print("\\nProcesso SWPC concluído com sucesso.")

    except Exception as e:
        print(f"Erro crítico na conexão FTP: {e}")

In [ ]:
download_swpc_ftp_events(DATA_RANGE, SAVE_PATH)

## SSW

In [11]:
def download_ssw_events(years_range_: range, save_path_: str) -> None:
    """
    Faz scrape do archive SSW em dois passos:
    1) lê a página principal com os snapshots;
    2) acessa cada snapshot e extrai a tabela de eventos.
    """
    archive_url = "https://www.lmsal.com/solarsoft/latest_events_archive.html"
    save_path_abs = os.path.abspath(save_path_)
    os.makedirs(save_path_abs, exist_ok=True)

    print(f"Arquivos SSW serão salvos em: '{save_path_abs}' (organizados por ano/SSW)")
    session = requests.Session()

    def _cell_text(tag) -> str:
        return tag.get_text(" ", strip=True) if tag is not None else ""

    def _find_table_by_headers(soup: BeautifulSoup, expected_headers: list[str]):
        for table in soup.find_all("table"):
            first_row = table.find("tr")
            if first_row is None:
                continue

            headers = [
                _cell_text(cell)
                for cell in first_row.find_all(["th", "td"], recursive=False)
            ]

            # Verifica se a tabela tem pelo menos a quantidade de colunas esperada
            if len(headers) < len(expected_headers):
                continue

            # Verifica se o texto esperado está contido no cabeçalho real (ignora sufixos extras)
            is_match = all(
                expected in actual
                for expected, actual in zip(expected_headers, headers)
            )

            if is_match:
                return table

        return None

    def _year_from_href(href: str) -> int | None:
        match = re.search(r"last_events_(\d{4})", href or "")
        return int(match.group(1)) if match else None

    def _snapshot_id_from_href(href: str) -> str:
        match = re.search(r"last_events_(\d{8}_\d{4})", href or "")
        if match:
            return match.group(1)
        return re.sub(r"[^0-9A-Za-z_.-]", "_", (href or "").rstrip("/").split("/")[-1]) or "snapshot"

    try:
        response = session.get(archive_url, timeout=30)
        response.raise_for_status()
    except requests.exceptions.RequestException as e:
        print(f"Erro ao acessar a página principal do SSW: {e}")
        return

    soup_archive = BeautifulSoup(response.text, "html.parser")
    archive_headers = ["Snapshot Time", "First Event", "Last Event", "Number of Events", "Largest Event", "#B", "#C", "#M", "#X"]
    archive_table = _find_table_by_headers(soup_archive, archive_headers)
    if archive_table is None:
        print("Erro: tabela principal do archive SSW não foi encontrada.")
        return

    snapshot_entries = []
    for row in archive_table.find_all("tr"):
        cells = row.find_all("td", recursive=False)
        if len(cells) != 9:
            continue
        link = cells[0].find("a", href=True)
        if link is None:
            continue
        year = _year_from_href(link["href"])
        if year is None or year not in years_range_:
            continue
        snapshot_entries.append(
            {
                "year": year,
                "snapshot_time": _cell_text(cells[0]),
                "href": link["href"],
            }
        )

    if not snapshot_entries:
        print("Nenhum snapshot encontrado dentro do intervalo de anos solicitado.")
        return

    event_headers = ["Event#", "EName", "Start", "Stop", "Peak", "GOES Class", "Derived Position"]
    total_snapshots = 0
    total_events = 0

    for snapshot in snapshot_entries:
        year = snapshot["year"]
        snapshot_time = snapshot["snapshot_time"]
        snapshot_url = urljoin(archive_url, snapshot["href"])
        snapshot_id = _snapshot_id_from_href(snapshot["href"])

        year_dir = os.path.join(save_path_abs, str(year), "SSW")
        os.makedirs(year_dir, exist_ok=True)
        output_path = os.path.join(year_dir, f"SSW_{snapshot_id}.csv")

        if os.path.exists(output_path):
            print(f"  [SKIP] {snapshot_time} -> arquivo já existe: {output_path}")
            continue

        print(f"  [SSW] Acessando snapshot {snapshot_time}: {snapshot_url}")
        try:
            snapshot_response = session.get(snapshot_url, timeout=30)
            snapshot_response.raise_for_status()
        except requests.exceptions.RequestException as e:
            print(f"  [SSW] Falha ao acessar snapshot {snapshot_url}: {e}")
            continue

        soup_snapshot = BeautifulSoup(snapshot_response.text, "html.parser")
        event_table = _find_table_by_headers(soup_snapshot, event_headers)
        if event_table is None:
            print(f"  [SSW] Tabela de eventos não encontrada em {snapshot_url}")
            continue

        events_rows = []
        for row in event_table.find_all("tr"):
            cells = row.find_all("td", recursive=False)
            if len(cells) != 7:
                continue

            event_no = _cell_text(cells[0])
            if not event_no.isdigit():
                continue

            events_rows.append(
                {
                    "Event#": int(event_no),
                    "EName": _cell_text(cells[1]),
                    "Start": _cell_text(cells[2]),
                    "Stop": _cell_text(cells[3]),
                    "Peak": _cell_text(cells[4]),
                    "GOES Class": _cell_text(cells[5]),
                    "Derived Position": _cell_text(cells[6]),
                }
            )

        if not events_rows:
            print(f"  [SSW] Snapshot sem eventos válidos: {snapshot_url}")
            continue

        pd.DataFrame(events_rows).to_csv(output_path, index=False, encoding="utf-8-sig")
        total_snapshots += 1
        total_events += len(events_rows)
        print(f"  [SSW] Salvo {len(events_rows)} eventos em: {output_path}")
        time.sleep(0.1)

    print(f"\nProcesso SSW concluído. Snapshots processados: {total_snapshots}; eventos extraídos: {total_events}.")

In [12]:
def organize_ssw_events_monthly(raw_root_dir: str, years_range_: range) -> None:
    """
    Move SSW snapshot CSVs from raw/<year>/SSW/ into monthly folders raw/<year>/SSW/01..12.

    Expected filename format: SSW_YYYYMMDD_HHMM.csv
    """
    raw_root_abs = os.path.abspath(raw_root_dir)
    month_pattern = re.compile(r"^SSW_(\d{4})(\d{2})(\d{2})_\d{4}\.csv$")

    for year in years_range_:
        ssw_dir = os.path.join(raw_root_abs, str(year), "SSW")
        if not os.path.isdir(ssw_dir):
            continue

        for file_name in os.listdir(ssw_dir):
            source_path = os.path.join(ssw_dir, file_name)
            if not os.path.isfile(source_path):
                continue

            match = month_pattern.match(file_name)
            if match is None:
                continue

            month = match.group(2)
            month_dir = os.path.join(ssw_dir, month)
            os.makedirs(month_dir, exist_ok=True)

            destiny_path = os.path.join(month_dir, file_name)
            if os.path.exists(destiny_path):
                continue

            shutil.move(source_path, destiny_path)

    return None

In [13]:
download_ssw_events(DATA_RANGE, SAVE_PATH)
organize_ssw_events_monthly(SAVE_PATH, DATA_RANGE)

Arquivos SSW serão salvos em: 'G:\My Drive\Solar_Flares\Data\events\raw' (organizados por ano/SSW)
  [SKIP] 31-Dec-2012 23:22 -> arquivo já existe: G:\My Drive\Solar_Flares\Data\events\raw\2012\SSW\SSW_20121231_2322.csv
  [SKIP] 30-Dec-2012 23:22 -> arquivo já existe: G:\My Drive\Solar_Flares\Data\events\raw\2012\SSW\SSW_20121230_2322.csv
  [SKIP] 29-Dec-2012 23:23 -> arquivo já existe: G:\My Drive\Solar_Flares\Data\events\raw\2012\SSW\SSW_20121229_2323.csv
  [SKIP] 28-Dec-2012 23:22 -> arquivo já existe: G:\My Drive\Solar_Flares\Data\events\raw\2012\SSW\SSW_20121228_2322.csv
  [SKIP] 27-Dec-2012 23:22 -> arquivo já existe: G:\My Drive\Solar_Flares\Data\events\raw\2012\SSW\SSW_20121227_2322.csv
  [SKIP] 26-Dec-2012 23:22 -> arquivo já existe: G:\My Drive\Solar_Flares\Data\events\raw\2012\SSW\SSW_20121226_2322.csv
  [SKIP] 25-Dec-2012 23:23 -> arquivo já existe: G:\My Drive\Solar_Flares\Data\events\raw\2012\SSW\SSW_20121225_2323.csv
  [SKIP] 24-Dec-2012 11:25 -> arquivo já existe: G:\My

## NCEI Locations

In [ ]:
def download_flloc_nc_data_selenium(years_range_: range, save_path_: str) -> None:
    """
    Baixa os dados de localização (flloc) da NCEI via Selenium.
    Restringe rigorosamente o download para datas a partir de 2017-02-09.
    """
    save_path_abs = os.path.abspath(save_path_)
    os.makedirs(save_path_abs, exist_ok=True)
    print(f"Arquivos FLLOC (Locations) serão salvos em: '{save_path_abs}' (organizados por ano/NCEI_FLLOC/mes)")

    options = Options()
    options.set_preference("browser.download.folderList", 2)
    options.set_preference("browser.download.dir", save_path_abs)
    options.set_preference("browser.download.manager.showWhenStarting", False)
    options.set_preference("browser.helperApps.neverAsk.saveToDisk", "text/csv,application/octet-stream,application/x-netcdf")

    driver = webdriver.Firefox(service=Service(GeckoDriverManager().install()), options=options)

    try:
        base_url = "https://data.ngdc.noaa.gov/platforms/solar-space-observing-satellites/goes/goes16/l2/data/xrsf-l2-flloc_science/"
        schedule: dict[int, set[str]] = {}

        for y_ in years_range_:
            if y_ < 2017:
                continue

            for date_ in pd.date_range(start=f"{y_}-01-01", end=f"{y_}-12-31"):
                if date_ < pd.Timestamp("2017-02-09"):
                    continue
                schedule.setdefault(y_, set()).add(f"{date_.month:02d}/")

        for y_, target_months in sorted(schedule.items()):
            print(f"Acessando diretório FLLOC do ano {y_}...")
            year_url = f"{base_url}{y_}/"
            driver.get(year_url)
            time.sleep(2)

            month_elements = driver.find_elements(By.XPATH, "//table/tbody/tr/td/a")
            month_links = {}
            for link in month_elements:
                text = link.text.strip()
                if re.fullmatch(r"\d{2}/", text):
                    month_links[text] = link.get_attribute("href")

            for month_dir in sorted(target_months):
                month_link = month_links.get(month_dir)
                if not month_link:
                    continue

                month_num = month_dir.rstrip('/')
                month_save_dir = os.path.join(save_path_abs, str(y_), "NCEI_FLLOC", month_num)
                os.makedirs(month_save_dir, exist_ok=True)

                print(f"  [MÊS] Acessando {y_}/{month_num}")
                driver.get(month_link)
                time.sleep(1)

                day_elements = driver.find_elements(By.XPATH, "//table/tbody/tr/td/a")
                for file in day_elements:
                    file_name_ = file.text.strip()
                    if 'Parent Directory' in file_name_ or not file_name_.endswith('.nc'):
                        continue

                    match_ = re.search(r'_d(\d{8})_', file_name_)
                    if match_:
                        file_date = pd.Timestamp(match_.group(1))
                        if file_date < pd.Timestamp("2017-02-09"):
                            continue

                    root_download_file_path = os.path.join(save_path_abs, file_name_)
                    month_file_path = os.path.join(month_save_dir, file_name_)

                    if os.path.exists(month_file_path):
                        continue

                    print(f"        -> Baixando {file_name_}...")
                    file.click()
                    util.wait_download(file_path=root_download_file_path, file_name=file_name_)
                    os.replace(root_download_file_path, month_file_path)

    except Exception as e:
        print(f"Erro na extração NCEI FLLOC: {e}")
    finally:
        print("\nProcesso NCEI FLLOC concluído.")
        driver.quit()

In [ ]:
download_flloc_nc_data_selenium(DATA_RANGE, SAVE_PATH)

## SRS Reports

In [ ]:
def download_swpc_ftp_srs(years_range_: range, save_path_: str) -> None:
    """
    Conecta ao servidor FTP da SWPC e baixa os relatórios diários
    de Regiões Ativas (SRS), extraindo-os para pastas anuais.
    """
    save_path_abs = os.path.abspath(save_path_)
    os.makedirs(save_path_abs, exist_ok=True)

    print(f"Arquivos SRS serão salvos em subpastas anuais de: '{save_path_abs}'")

    ftp_host = "ftp.swpc.noaa.gov"
    base_dir = "/pub/warehouse/"

    try:
        print(f"Estabelecendo conexão FTP com: {ftp_host} para arquivos SRS...")
        ftp = ftplib.FTP(ftp_host)
        ftp.login()

        for y_ in years_range_:
            year_dir_ = f"{base_dir}{y_}/"
            expected_filename = f"{y_}_SRS.tar.gz"

            year_save_dir = os.path.join(save_path_abs, str(y_))
            extracted_dir = os.path.join(year_save_dir, f"{y_}_SWPC_SRS")
            archive_path = os.path.join(year_save_dir, expected_filename)

            os.makedirs(year_save_dir, exist_ok=True)

            if os.path.isdir(extracted_dir) and os.listdir(extracted_dir):
                print(f"  -> [SKIP] Pasta SRS já extraída: {extracted_dir}")
                continue

            try:
                ftp.cwd(year_dir_)
            except ftplib.error_perm as e:
                print(f"  -> AVISO: Diretório do ano {y_} não encontrado. Erro: {e}")
                continue

            files_in_dir = ftp.nlst()
            if expected_filename not in files_in_dir:
                print(f"  -> AVISO: O arquivo {expected_filename} não foi encontrado na pasta {year_dir_}.")
                continue

            if not os.path.exists(archive_path):
                print(f"  -> Baixando {expected_filename} via FTP...")
                with open(archive_path, "wb") as f:
                    ftp.retrbinary(f"RETR {expected_filename}", f.write)
            else:
                print(f"  -> Reutilizando arquivo SRS já baixado: {archive_path}")

            os.makedirs(extracted_dir, exist_ok=True)
            print(f"  -> Extraindo arquivos SRS para: {extracted_dir}")
            with tarfile.open(archive_path, "r:gz") as tar:
                tar.extractall(path=extracted_dir)

            os.remove(archive_path)
            time.sleep(1)

        ftp.quit()
        print("\nProcesso SWPC SRS concluído com sucesso.")

    except Exception as e:
        print(f"Erro crítico na conexão FTP para SRS: {e}")

In [ ]:
download_swpc_ftp_srs(DATA_RANGE, SAVE_PATH)

## Check Files

In [ ]:
def check_raw_data_integrity(years_range_: range, save_path_: str):
    """
    Varre o diretório raw e verifica a existência de todos os arquivos
    diários necessários para cada catálogo, identificando buracos temporais.
    """
    save_path_abs = os.path.abspath(save_path_)
    print(f"Iniciando validação de integridade em: {save_path_abs}\n")

    missing_report = {
        "Science_Quality_NC": [],
        "SWPC_Events": [],
        "SSW_Events": [],
        "NCEI_FLLOC": [],
        "SRS_Reports": []
    }

    dir_cache = {}
    def get_files_in_dir(dp: str) -> list[str]:
        if dp not in dir_cache:
            try:
                dir_cache[dp] = os.listdir(dp)
            except FileNotFoundError:
                dir_cache[dp] = []
        return dir_cache[dp]

    start_date = f"{years_range_.start}-01-01"
    end_date = f"{years_range_.stop - 1}-12-31"

    for date_ in pd.date_range(start=start_date, end=end_date):
        y_str = str(date_.year)
        m_str = f"{date_.month:02d}"
        d_str = f"{date_.day:02d}"
        ymd_str = f"{y_str}{m_str}{d_str}"

        # 1. Base Primária: Science-Quality NC
        try:
            sat, _ = get_events_source(date_)
            expected_prefix_nc = f"sci_xrsf-l2-flsum_{sat}_d{ymd_str}"
            nc_dir = os.path.join(save_path_abs, y_str, m_str)
            nc_files = get_files_in_dir(nc_dir)
            if not any(f.startswith(expected_prefix_nc) for f in nc_files):
                missing_report["Science_Quality_NC"].append(ymd_str)
        except ValueError:
            pass

        # 2. SWPC FTP (Operacional)
        swpc_dir = os.path.join(save_path_abs, y_str, f"{y_str}_SWPC_events")
        swpc_files = get_files_in_dir(swpc_dir)
        expected_swpc = f"{ymd_str}events.txt"
        if expected_swpc not in swpc_files:
            missing_report["SWPC_Events"].append(ymd_str)

        # 3. SSW Events
        ssw_dir = os.path.join(save_path_abs, y_str, "SSW", m_str)
        ssw_files = get_files_in_dir(ssw_dir)
        expected_prefix_ssw = f"SSW_{ymd_str}"
        if not any(f.startswith(expected_prefix_ssw) for f in ssw_files):
            missing_report["SSW_Events"].append(ymd_str)

        # 4. NCEI FLLOC (Somente a partir de 2017-02-09)
        if date_ >= pd.Timestamp("2017-02-09"):
            flloc_dir = os.path.join(save_path_abs, y_str, "NCEI_FLLOC", m_str)
            flloc_files = get_files_in_dir(flloc_dir)
            expected_prefix_flloc = f"sci_xrsf-l2-flloc_g16_d{ymd_str}"
            if not any(f.startswith(expected_prefix_flloc) for f in flloc_files):
                missing_report["NCEI_FLLOC"].append(ymd_str)

        # 5. SRS Reports
        srs_dir = os.path.join(save_path_abs, y_str, f"{y_str}_SWPC_SRS")
        srs_files = get_files_in_dir(srs_dir)
        expected_srs = f"{ymd_str}SRS.txt"
        if expected_srs not in srs_files:
            missing_report["SRS_Reports"].append(ymd_str)

    print("=== RELATÓRIO DE INTEGRIDADE ===")
    for key, missing_list in missing_report.items():
        qtd = len(missing_list)
        if qtd == 0:
            print(f"[OK] {key}: 100% completo.")
        else:
            print(f"[ALERTA] {key}: Faltam {qtd} dias no dataset.")
            print(f"         Exemplos ausentes: {missing_list[:5]}...")

    return missing_report

In [ ]:
missing_report = check_raw_data_integrity(DATA_RANGE, SAVE_PATH)

In [ ]:
missing_report['Science_Quality_NC']